In [ ]:
from reflect.llm.prompter import PortkeyLLMPrompter
from reflect.pipelines.fast_validation import (
    scene_graphs_mem, summary_mem, run_reasoning_mem,
    generate_replan_multi_plan_sim,
)
from reflect.core.data import load_episode_data
from reflect.perception.audio import process_sound

from matplotlib import pyplot as plt
from matplotlib.patches import Patch

import os, json
import numpy as np
from IPython.display import display, HTML


In [ ]:
PORTKEY_API_KEY = os.environ["PORTKEY_API_KEY"]  # set in your shell or .env (see .env.example)
MODEL = "gpt-5.4"
REASONING_EFFORT = "none"
from reflect.core.paths import prompts_dir, sim_data_root

ROOT = str(sim_data_root()) + "/"  # honors REFLECT_DATA_ROOT
WITH_AUDIO = 1
PROMPT_TEMPLATE = str(prompts_dir() / "sim_prompts.json")

In [ ]:
CACHE_DIR = os.path.join(ROOT, ".llm_cache", MODEL)
prompter = PortkeyLLMPrompter(
    portkey_api_key=PORTKEY_API_KEY,
    model=MODEL,
    reasoning_effort=REASONING_EFFORT,
    cache_dir=CACHE_DIR,
)
print(f"LLM cache dir: {CACHE_DIR}")

In [ ]:
task_name = "boilWater"
episode_name = "boilWater-10"
data_path = os.path.join(ROOT, task_name, episode_name)

# Load episode data
with open(os.path.join(data_path, "task.json"), "r") as f:
    task_detail = json.load(f)

# Load prompt template
with open(PROMPT_TEMPLATE, "r") as f:
    prompt_template = json.load(f)

events, task_detail, object_list, interact_actions, nav_actions = load_data(data_path, task_detail)

In [ ]:
# Sound detection
detected_sounds = process_sound(data_path, object_list)

In [ ]:
# Generate scene graphs for each step in the episode
local_sgs, global_sg, key_frames = scene_graphs_mem(events, object_list,
                                                    nav_actions, interact_actions,
                                                    WITH_AUDIO, detected_sounds, task_detail)

In [ ]:
# Generate summaries
l1, l2, l1_captions, l2_captions = summary_mem(events, nav_actions, interact_actions,
                                               WITH_AUDIO, detected_sounds, task_detail,
                                               local_sgs, key_frames)

In [ ]:
# Run reasoning
reasoning_dict, reasoning_prompts = run_reasoning_mem(task_detail,
                                                        prompter,
                                                        global_sg,
                                                        prompt_template,
                                                        list(l2_captions),
                                                        list(l1_captions),
                                                        data_path=data_path,
                                                        task_name=task_name,
                                                        episode_name=episode_name)

In [ ]:
gen_prompter = PortkeyLLMPrompter(
    portkey_api_key=PORTKEY_API_KEY,
    model=MODEL,
    reasoning_effort="medium",
    cache_dir=CACHE_DIR,
)

replan_dict, correction_prompts = generate_replan_multi_plan_sim(
    data_path=data_path,
    task=task_detail,
    final_event=events[-1],
    last_frame=len(events) - 1,
    object_list=object_list,
    gen_prompter=gen_prompter,
    score_prompter=prompter,
    prompt_info=prompt_template,
    global_sg=global_sg,
    pred_failure_reason=reasoning_dict["pred_failure_reason"],
    task_name=task_name,
    max_steps=10,
    num_candidates=4,
)
print(f"Success: {replan_dict['success']}")


In [ ]:
replan_dict["success"]

In [ ]:
for prompt in correction_prompts:
    print(f"--- {prompt['call']} (step {prompt['step_index']}) ---")
    print("Prompt:")
    for role, text in prompt['prompt'].items():
        print(f"{role.upper()}: {text}")
    print("Response:", prompt['response'])
    if 'score' in prompt:
        print("Score:", prompt['score'])
    print("\n")

In [ ]:
prompts_log = reasoning_prompts + correction_prompts

In [ ]:
print(f"Selected correction plan ({replan_dict['num_steps']} steps):")
for i, step in enumerate(replan_dict["selected_plan"], 1):
    print(f"  {i}. {step}")
print()
print(f"Original task plan ({len(replan_dict['task_plan'])} steps):")
for i, step in enumerate(replan_dict["task_plan"], 1):
    print(f"  {i}. {step}")

In [ ]:
plan_trace = replan_dict["plan_trace"]
step_indices   = [e["step_index"]    for e in plan_trace]
confidences_p  = [e.get("confidence") for e in plan_trace]
entropies_p    = [e.get("entropy")    for e in plan_trace]
sel_labels     = [e.get("selected_label") for e in plan_trace]

neg_log_conf_p = [-np.log(c) if c is not None and c > 0 else 10 for c in confidences_p]
done_flags     = [e.get("done", False) for e in plan_trace]

# Color: green = action selected, grey = DONE step
bar_colors_p = ["#aec7e8" if not d else "#cccccc" for d in done_flags]

legend_handles_p = [
    Patch(facecolor="#aec7e8", edgecolor="black", label="Action selected"),
    Patch(facecolor="#cccccc", edgecolor="black", label="DONE (no action)"),
]

fig, axes = plt.subplots(1, 2, figsize=(max(10, len(step_indices) * 2), 5))

# Left: -log(confidence)
ax = axes[0]
bars = ax.bar(range(len(step_indices)), neg_log_conf_p, color=bar_colors_p, edgecolor="black", linewidth=0.6)
for i, (bar, conf) in enumerate(zip(bars, confidences_p)):
    lbl = f"{conf:.6f}" if conf is not None else "N/A"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0002,
            lbl, ha="center", va="bottom", fontsize=7, fontweight="bold", rotation=45)
ax.set_xticks(range(len(step_indices)))
ax.set_xticklabels([f"Step {i}\n{sel_labels[i] or ''}" for i in range(len(step_indices))],
                   rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Plan step")
ax.set_ylabel("−log(confidence)")
ax.set_title("Replan Selection Surprise per Step (−log conf)")
ax.legend(handles=legend_handles_p, fontsize=7)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# Right: entropy
safe_ent_p = [e if e is not None else 0.0 for e in entropies_p]
ax2 = axes[1]
bars2 = ax2.bar(range(len(step_indices)), safe_ent_p, color=bar_colors_p, edgecolor="black", linewidth=0.6)
for i, (bar, ent) in enumerate(zip(bars2, entropies_p)):
    lbl = f"{ent:.4f}" if ent is not None else "N/A"
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0001,
             lbl, ha="center", va="bottom", fontsize=7, fontweight="bold", rotation=45)
ax2.set_xticks(range(len(step_indices)))
ax2.set_xticklabels([f"Step {i}\n{sel_labels[i] or ''}" for i in range(len(step_indices))],
                    rotation=45, ha="right", fontsize=8)
ax2.set_xlabel("Plan step")
ax2.set_ylabel("Normalized Entropy")
ax2.set_title("Replan Selection Entropy per Step")
ax2.legend(handles=legend_handles_p, fontsize=7)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
detector_trace = reasoning_dict.get("detector_trace")
step_trace = []
for step in detector_trace:
    step_trace.append({
        "step": step["step"],
        "subgoal": step["subgoal"],
        "predicted_label": step["predicted_label"],
        "confidence": (step["score"] or {}).get("confidence"),
    })

# Ground truth failure step directly from task metadata (e.g. "00:36")
gt_failure_step = task_detail.get("gt_failure_step")

# Predicted failure step(s) from reasoning output
pred_failure_steps = reasoning_dict.get("pred_failure_step", [])


In [ ]:
def _ts_to_sec(ts):
    m, s = str(ts).strip().split(":")
    return int(m) * 60 + int(s)

steps = [step["step"] for step in step_trace]
x_vals = [_ts_to_sec(s) for s in steps]

plt.figure(figsize=(10, 5))
plt.plot(x_vals, [step["confidence"] for step in step_trace], marker='o')
plt.title("Detector Confidence Over Steps")
plt.xlabel("Time (seconds)")
plt.ylabel("Confidence")

# Vertical line at the ground truth failure step
if gt_failure_step is not None:
    # gt_failure_step may be a string "MM:SS" or a list/nested list
    if isinstance(gt_failure_step, list):
        flat = gt_failure_step[0] if gt_failure_step else None
        gt_ts = flat[0] if isinstance(flat, list) else flat
    else:
        gt_ts = gt_failure_step
    if gt_ts:
        plt.axvline(x=_ts_to_sec(gt_ts), color='red', linestyle='--', label=f'Ground Truth Failure ({gt_ts})')

# Vertical lines at predicted failure steps
for pred_step in pred_failure_steps:
    try:
        plt.axvline(x=_ts_to_sec(pred_step), color='blue', linestyle=':', label=f'Predicted Failure ({pred_step})')
    except (ValueError, AttributeError):
        pass

plt.xticks(ticks=x_vals, labels=steps, rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


In [ ]:

def _sec_to_bar_pos(sec, xv):
    """Map seconds to a fractional bar-chart position via linear interpolation."""
    if not xv:
        return 0.0
    if sec <= xv[0]:
        return 0.0
    if sec >= xv[-1]:
        return float(len(xv) - 1)
    for i in range(len(xv) - 1):
        if xv[i] <= sec <= xv[i + 1]:
            span = xv[i + 1] - xv[i]
            return i + ((sec - xv[i]) / span if span > 0 else 0.0)
    return float(len(xv) - 1)

# Re-derive gt_ts here so this cell is self-contained
_gt_fs = task_detail.get("gt_failure_step")
if isinstance(_gt_fs, list):
    _gt_fs = _gt_fs[0]
if isinstance(_gt_fs, list):
    _gt_fs = _gt_fs[0]
gt_ts_local = _gt_fs

# Extract per-step scores
confidences = [step["confidence"] for step in step_trace]
entropies = [(step["score"] or {}).get("entropy") for step in detector_trace]
subgoal_labels = [step["subgoal"] for step in step_trace]
predicted_labels = [step["predicted_label"] for step in step_trace]

# Use negative log-confidence as the uncertainty metric - it reveals
# differences that are invisible on a linear (1−conf) scale.
# −log(0.9999) ≈ 0.0001  vs  −log(0.9997) ≈ 0.0003  → visible gap
neg_log_conf = [-np.log(c) if c is not None and c > 0 else 10 for c in confidences]

# Color bars by predicted label
bar_colors = ["#d62728" if lbl != "A" else "#1f77b4" for lbl in predicted_labels]

legend_handles = [
    Patch(facecolor="#d62728", edgecolor="black", label="Predicted: B (failure)"),
    Patch(facecolor="#1f77b4", edgecolor="black", label="Predicted: A (success)"),
]

safe_entropies = [e if e is not None else 0.0 for e in entropies]
plt.bar(range(len(x_vals)), safe_entropies, color=bar_colors, edgecolor="black", linewidth=0.6)

extra_handles2 = []
if gt_ts_local:
    gt_sec = _ts_to_sec(gt_ts_local)
    bar_pos = _sec_to_bar_pos(gt_sec, x_vals)
    line2 = plt.axvline(x=bar_pos, color="red", linestyle="--", linewidth=1.5, label=f"GT Failure ({gt_ts_local})")
    extra_handles2.append(line2)
for pred_step in pred_failure_steps:
    try:
        ps = _ts_to_sec(pred_step)
        bar_pos = _sec_to_bar_pos(ps, x_vals)
        line2 = plt.axvline(x=bar_pos, color="blue", linestyle=":", linewidth=1.5, label=f"Pred Failure ({pred_step})")
        extra_handles2.append(line2)
    except (ValueError, AttributeError):
        pass

plt.xticks(range(len(x_vals)), [f"{steps[i]}\n{subgoal_labels[i][:25]}" for i in range(len(steps))],
           rotation=45, ha="right", fontsize=7)
plt.xlabel("Step (timestamp / subgoal)")
plt.ylabel("Normalized Entropy")
plt.title("Entropy per Step")
plt.legend(handles=legend_handles + extra_handles2, fontsize=7)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()


In [ ]:
# Diagnostic: inspect raw score metadata for each subgoal step
for i, entry in enumerate(detector_trace):
    score = entry.get("score") or {}
    print(f"Step {entry['step']} | label={entry['predicted_label']} | "
          f"status={score.get('score_status')} | "
          f"fallback={score.get('fallback_applied')} | "
          f"confidence={score.get('confidence')} | "
          f"entropy={score.get('entropy')} | "
          f"option_probs={score.get('option_probs')} | "
          f"fallback_logprob={score.get('fallback_logprob')}")

In [ ]:
prompt_items = []
for i, correction_prompt in enumerate(correction_prompts):
    prompt_type = correction_prompt["call"]
    system = correction_prompt["prompt"]["system"]
    user = correction_prompt["prompt"]["user"]
    response = correction_prompt["response"]
    options_conf_html = ""
    # Show confidence for each option if available
    if "options" in correction_prompt:
        options = correction_prompt["options"]
        option_probs = correction_prompt.get("option_probs")
        if option_probs and isinstance(option_probs, dict):
            options_conf_html += "<b>Option Confidences:</b><ul>"
            for opt, conf in option_probs.items():
                options_conf_html += f"<li><b>{opt}:</b> {conf:.4f}</li>"
            options_conf_html += "</ul>"
        elif isinstance(options, list):
            options_conf_html += "<b>Options:</b> " + ", ".join(str(o) for o in options)
    prompt_html = f'''
    <details style="margin-bottom:10px;">
      <summary style="font-weight:bold;">Correction Prompt {i} ({prompt_type})</summary>
      <div style="margin-left:15px;">
        <b>System:</b><pre style="white-space:pre-wrap;">{system}</pre>
        <b>User:</b><pre style="white-space:pre-wrap;">{user}</pre>
        {options_conf_html}
        <b>Response:</b><pre style="white-space:pre-wrap;">{response}</pre>
      </div>
    </details>
    '''
    prompt_items.append(prompt_html)

full_html = "\n".join(prompt_items)
display(HTML(full_html))

In [ ]:
for id, correction_prompt in enumerate(correction_prompts):
    print(f"--- Correction Prompt {id} ---")
    print(f"Prompt type: {correction_prompt['call']}")
    print(f"System prompt:\n{correction_prompt['prompt']['system']}\n")
    print(f"User prompt:\n{correction_prompt['prompt']['user']}\n")